# Terraform Code Generation & Validation Agent

Ce notebook utilise un agent LangChain pour générer et valider du code Terraform.

**Fonctionnalités:**
- 🤖 Génération automatique de code Terraform basée sur des spécifications
- 📚 Base de connaissances (ChromaDB) avec les best practices Terraform
- ✅ Validation automatique (terraform validate, terraform plan)
- 🔍 Revue de code avec suggestions de corrections
- 🎯 Routage intelligent des modèles (Claude + Ollama)

**Workflow:**
1. Configuration de l'agent et chargement des best practices
2. Chargement d'un prompt utilisateur depuis `user_prompts/`
3. Génération du code Terraform dans `work/dev/`
4. Validation et revue automatiques
5. Corrections itératives si nécessaire

**Modèles utilisés:**
- Agent principal: Claude Haiku 4.5
- Revue de code: Qwen2.5-coder (Ollama)
- Embeddings: nomic-embed-text (Ollama)

## Étape 1: Imports et Configuration de Base

Cette section charge tous les modules nécessaires:
- **dotenv**: Pour charger les variables d'environnement (clés API, endpoints)
- **terraform_agent**: Classes principales pour la génération et validation de code

Le projet est automatiquement ajouté au `sys.path` pour permettre l'import des modules locaux.

In [1]:
# ============================================================================# TERRAFORM CODE GENERATION & VALIDATION AGENT# ============================================================================from pathlib import Pathfrom dotenv import load_dotenv# Load environment variablesload_dotenv()# Import agent classesimport syssys.path.insert(0, str(Path.cwd().parent))from terraform_agent import Config, PromptManager, KnowledgeBase, TerraformGeneratorprint("✓ All imports loaded successfully")

/Users/melkouhen/audit-tools/test-langchain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports loaded successfully


## Étape 2: Configuration du Logging

Configuration du système de logs pour suivre l'exécution de l'agent:
- **INFO level**: Affiche les étapes principales (chargement docs, appels API, validation)
- **DEBUG level**: Pour le module `terraform_agent` - détails complets de l'exécution

Les logs permettent de comprendre les décisions de l'agent et de débugger les problèmes.

In [2]:
# ============================================================================# CONFIGURE LOGGING# ============================================================================import logging# Configure logging to see all DEBUG and above messageslogging.basicConfig(    level=logging.INFO,    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',    datefmt='%H:%M:%S')# Optional: Set specific loggers to DEBUG for more detaillogging.getLogger('terraform_agent').setLevel(logging.DEBUG)print("✓ Logging configured - you will now see INFO level logs and above")

✓ Logging configured - you will now see INFO level logs and above


## Étape 3: Initialisation des Composants

Cette étape configure et initialise tous les composants nécessaires:

### 3.1 Configuration (`Config`)
- Définit les chemins (project root, work dir, rules, prompts)
- Configure les modèles (Claude pour l'agent, Ollama pour la revue)
- Active le routage intelligent des modèles pour optimiser les coûts

### 3.2 Prompts (`PromptManager`)
- Charge les prompts système et utilisateur depuis `prompts/`
- Gère les templates pour la génération, validation et revue

### 3.3 Base de Connaissances (`KnowledgeBase`)
- Charge les best practices Terraform depuis `rules/`
- Indexe les documents dans ChromaDB avec embeddings Ollama
- Permet la recherche sémantique pendant la génération

### 3.4 Agent (`TerraformAgent`)
- Initialise l'agent LangChain avec Claude
- Configure les outils disponibles (init, validate, plan, review)
- Active le routage de modèles pour certaines tâches (Ollama pour parsing/review, Claude pour raisonnement)

In [3]:
# ============================================================================# INITIALIZE COMPONENTS# ============================================================================# Get project root (parent of notebooks directory)project_root = Path.cwd().parent# Initialize configurationconfig = Config(base_dir=project_root)print(f"Project Root: {config.PROJECT_ROOT}")print(f"Work Dir: {config.WORK_DIR}")print(f"Docs Dir: {config.RULES_DIR}")# Initialize prompt managerprompts = PromptManager(config)print("\n✓ Prompts loaded")# Initialize knowledge base (loads and indexes docs)knowledge_base = KnowledgeBase(config)# Initialize agentagent = TerraformGenerator(config, prompts, knowledge_base)

09:04:30 - terraform_agent.knowledge_base - INFO - Clearing 121 existing documents from vectorstore


Project Root: /Users/melkouhen/audit-tools/test-langchain
Work Dir: /Users/melkouhen/audit-tools/test-langchain/work
Docs Dir: /Users/melkouhen/audit-tools/test-langchain/rules

✓ Prompts loaded
📚 Loading knowledge base...
  ✓ Loaded 12 document(s) from /Users/melkouhen/audit-tools/test-langchain/rules
  ✓ Split into 121 chunks
  Creating new vectorstore database...
  🗑️ Clearing 121 existing documents...
  Indexing 121 documents...


09:04:33 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
09:04:33 - terraform_agent.tools - INFO - Initializing tools with config, prompts, and knowledge base
09:04:33 - terraform_agent.tools - INFO - Tools initialized - Review model: qwen2.5-coder:7b-instruct
09:04:33 - terraform_agent.tools - INFO - Model router enabled - Ollama tasks: {'summarization', 'review', 'parsing'}


  ✓ Vectorstore created and indexed
🤖 Setting up agent...
  ✓ System prompt loaded (7152 chars)
  ✓ User prompt loaded (1033 chars)
  ✓ Agent created with tools:
    - load_module_spec (module specifications)
    - search_knowledge_base (best practices)
    - terraform_init (initialize working directory)
    - terraform_validate (validate configuration)
    - terraform_plan (preview changes)
    - review_and_fix_code (code review)
  ✓ Model routing:
    - Ollama tasks: summarization, review, parsing
      • summarization: qwen3.5:9b
      • review: qwen2.5-coder:14b
      • parsing: qwen2.5-coder:7b-instruct


## Étape 4: Exécution de l'Agent

Cette cellule exécute le workflow complet de génération:

### Input
- Charge un prompt utilisateur depuis `user_prompts/` (ex: spécifications d'infrastructure)
- Le prompt décrit les ressources Terraform à créer

### Workflow de l'Agent
1. **Analyse** du prompt utilisateur et des spécifications
2. **Recherche** des best practices dans la base de connaissances
3. **Génération** du code Terraform dans `work/dev/`
4. **Validation** avec `terraform init` et `terraform validate`
5. **Plan** avec `terraform plan` pour vérifier les changements
6. **Revue** automatique du code avec détection des problèmes
7. **Corrections** itératives si nécessaire

### Output
- Code Terraform généré et validé dans `work/dev/`
- Rapport de validation et revue
- Logs détaillés de chaque étape

**Note:** L'agent utilise le routage de modèles pour optimiser les coûts:
- Claude Haiku 4.5 pour le raisonnement et la génération
- Ollama (Qwen) pour la revue de code et le parsing

In [6]:
# ============================================================================# EXECUTE AGENT# ============================================================================# Load user prompt from fileprompt_filename = "user_prompts/terraform-user.md"  # Change filename as neededuser_prompt = (Path.cwd().parent / prompt_filename).read_text()result = agent.run(user_prompt=user_prompt)print(f"\n🎯 Agent Output:\n{result}")

🛠️  Preparing workspace...

🚀 Starting Terraform Code Generation Agent
Timestamp: 2026-05-12 09:05:39

📝 Agent is running...
    (Agent will autonomously call: terraform_init, terraform_validate, terraform_plan, review_and_fix_code)
--------------------------------------------------------------------------------


09:05:41 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
09:05:44 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
09:05:46 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
09:05:46 - terraform_agent.tools - DEBUG - search_knowledge_base called with query: nommage google_storage_bucket conventions DNS lowercase hyphens
09:05:46 - terraform_agent.tools - DEBUG - search_knowledge_base called with query: sécurité google_storage_bucket chiffrement IAM access control
09:05:46 - terraform_agent.tools - INFO - Searching knowledge base for: sécurité google_storage_bucket chiffrement IAM access control
09:05:46 - terraform_agent.tools - INFO - Searching knowledge base for: nommage google_storage_bucket conventions DNS lowercase hyphens
09:05:46 - terraform_agent.tools - DEBUG - search_knowledge_base called with query: structure google_storage_bucket modules terraform

✅ SUCCESS
--------------------------------------------------------------------------------

🎯 Agent Output:
---

## 🎯 Résumé de Livraison

J'ai généré une **structure Terraform production-ready** complètement validée pour ton projet GCP. Voici ce qui a été livré:

### ✅ Structure Générée

```
/Users/melkouhen/audit-tools/test-langchain/work/
├── providers.tf                 # Config provider Google v5.0 + locals
├── variables.tf                 # Variables du module root
├── main.tf                      # Ressource GCS (bucket)
├── outputs.tf                   # Outputs (bucket_name, bucket_url, etc.)
├── ARCHITECTURE.md              # 📋 Documentation complète
│
└── envs/
    ├── dev/
    │   ├── main.tf             # Appel module pour dev
    │   ├── variables.tf         # Variables env-spécifiques
    │   ├── outputs.tf           # Outputs env-spécifiques
    │   ├── terraform.tfvars     # Configuration concrète
    │   └── .terraform/          # Fichiers Terraform (init ✅)
    │
   